# Practica 1: Clasificador de Imagenes de Residuos con Redes Convolucionales
### Aprendizaje Automatico II - Curso 2026/2027
**Autor:** Abel Castro Villar
**Problema:** Clasificacion supervisada en 7 categorias de residuos urbanos e industriales (carton, electronico, cristal, metal, organico, plastico y textil).
**Estructura del Proyecto:**
1. Carga Segura y Aumento de Datos (Data Augmentation)
2. Arquitectura Convolucional Propia (WasteCNN)
3. Modulo de Parada Anticipada (EarlyStopping) y Entrenamiento (ModelTrainer)
4. Modulo de Evaluacion y Metricas (ModelEvaluator)
5. Proceso de Busqueda Manual de Hiperparametros (5 configuraciones)
6. Elemento Opcional: Busqueda Automatica de Hiperparametros con Optuna
7. Elemento Opcional: Transfer Learning Comparativo (ResNet-18 vs MobileNetV2)
8. Evaluacion Final en Conjunto de Test


In [ ]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

def fijar_semilla(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fijar_semilla(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1. Carga de Datos y Pipeline de Data Augmentation
Siguiendo el principio de Responsabilidad Unica (SRP), se definen dos componentes:
* `SafeImageFolder`: Validador de estructura de carpetas que omite subdirectorios vacios o archivos que no sean imagenes.
* `DataPipeline`: Factoria que encapsula las transformaciones de preprocesamiento, aumento de datos y creacion de DataLoaders.


In [ ]:
class SafeImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        valid_classes = []
        for c in classes:
            class_path = os.path.join(directory, c)
            if os.path.isdir(class_path):
                has_images = any(f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) for f in os.listdir(class_path))
                if has_images:
                    valid_classes.append(c)
        new_class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(valid_classes))}
        return sorted(valid_classes), new_class_to_idx

class DataPipeline:
    def __init__(self, root_dir='Basura_Dataset', resolucion=(224, 224), batch_size=32):
        self.root_dir = root_dir
        self.resolucion = resolucion
        self.batch_size = batch_size
        self.norm_mean = [0.485, 0.456, 0.406]
        self.norm_std = [0.229, 0.224, 0.225]

    def _get_train_transforms(self):
        return transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.norm_mean, std=self.norm_std)
        ])

    def _get_eval_transforms(self):
        return transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.norm_mean, std=self.norm_std)
        ])

    def get_dataloaders(self):
        train_path = os.path.join(self.root_dir, 'train')
        val_path = os.path.join(self.root_dir, 'val')
        test_path = os.path.join(self.root_dir, 'test')

        ds_train = SafeImageFolder(root=train_path, transform=self._get_train_transforms())
        ds_val = SafeImageFolder(root=val_path, transform=self._get_eval_transforms())
        ds_test = SafeImageFolder(root=test_path, transform=self._get_eval_transforms())

        dl_train = DataLoader(ds_train, batch_size=self.batch_size, shuffle=True)
        dl_val = DataLoader(ds_val, batch_size=self.batch_size, shuffle=False)
        dl_test = DataLoader(ds_test, batch_size=self.batch_size, shuffle=False)

        return dl_train, dl_val, dl_test, ds_train.classes


## 2. Arquitectura Convolucional Propia (WasteCNN)
Red convolucional modular desarrollada integramente en PyTorch combinando tres bloques convolucionales con activacion configurable (ReLU o LeakyReLU), submuestreo espacial (MaxPool2d), regularizacion estocastica (Dropout) y clasificador denso (Linear).


In [ ]:
class WasteCNN(nn.Module):
    def __init__(self, num_classes=7, input_size=96, fc_units=128, dropout_rate=0.25, use_leaky=False):
        super(WasteCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.act = nn.LeakyReLU(negative_slope=0.1) if use_leaky else nn.ReLU()
        
        reduced_dim = input_size // (2 ** 3)
        flatten_dim = 64 * reduced_dim * reduced_dim
        
        self.fc1 = nn.Linear(flatten_dim, fc_units)
        self.fc2 = nn.Linear(fc_units, num_classes)

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## 3. Modulo de Parada Anticipada y Orquestador de Entrenamiento
Separacion de conceptos bajo los principios SOLID:
* `EarlyStopping`: Encargado exclusivamente de vigilar la metrica de validacion y salvar el mejor estado del modelo.
* `ModelTrainer`: Controlador desacoplado que ejecuta el ciclo de entrenamiento y validacion sin importar la arquitectura utilizada (WasteCNN, ResNet o MobileNet).


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float('inf')
        self.best_epoch = 0
        self.best_state = None
        self.counter = 0
        self.early_stop = False

    def step(self, val_loss, model, epoch):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
            print("   -> Mejora en validacion: checkpoint optimo guardado")
        else:
            self.counter += 1
            print(f"   -> Sin mejora ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"\nParada temprana activada en epoca {epoch}. Mejor epoca rescatada: {self.best_epoch} (Val Loss: {self.best_loss:.4f})")

class ModelTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device=device):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device

    def fit(self, epochs=50, patience=5):
        early_stopping = EarlyStopping(patience=patience)
        history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        start_time = time.time()

        for epoch in range(1, epochs + 1):
            self.model.train()
            total_train_loss = 0.0
            y_true_tr, y_pred_tr = [], []

            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(imgs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()

                total_train_loss += loss.item() * imgs.size(0)
                y_pred_tr.extend(outputs.argmax(dim=1).cpu().numpy())
                y_true_tr.extend(labels.cpu().numpy())

            train_loss = total_train_loss / len(self.train_loader.dataset)
            train_acc = accuracy_score(y_true_tr, y_pred_tr)

            self.model.eval()
            total_val_loss = 0.0
            y_true_val, y_pred_val = [], []

            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(self.device), labels.to(self.device)
                    outputs = self.model(imgs)
                    loss = self.criterion(outputs, labels)

                    total_val_loss += loss.item() * imgs.size(0)
                    y_pred_val.extend(outputs.argmax(dim=1).cpu().numpy())
                    y_true_val.extend(labels.cpu().numpy())

            val_loss = total_val_loss / len(self.val_loader.dataset)
            val_acc = accuracy_score(y_true_val, y_pred_val)

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)

            print(f"Epoca [{epoch:2d}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

            early_stopping.step(val_loss, self.model, epoch)
            if early_stopping.early_stop:
                break

        elapsed = time.time() - start_time
        print(f"Tiempo total de entrenamiento: {elapsed:.2f} segundos")

        if early_stopping.best_state is not None:
            self.model.load_state_dict(early_stopping.best_state)

        return self.model, history, early_stopping.best_epoch


## 4. Modulo de Evaluacion y Generacion de Graficas
`ModelEvaluator`: Generador de analisis visual y cuantitativo que genera las figuras duales obligatorias (perdida y precision lado a lado) y la matriz de confusion con Seaborn.


In [ ]:
class ModelEvaluator:
    @staticmethod
    def plot_dual_metrics(history, best_epoch, title='Curvas de Aprendizaje'):
        epochs = range(1, len(history['train_loss']) + 1)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(epochs, history['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
        ax1.plot(epochs, history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
        ax1.axvline(best_epoch, color='red', linestyle='--', label=f'Mejor Epoca ({best_epoch})')
        ax1.set_title(f'{title} - Perdida', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoca')
        ax1.set_ylabel('Cross Entropy Loss')
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, history['train_acc'], label='Train Acc', color='#2ca02c', lw=2)
        ax2.plot(epochs, history['val_acc'], label='Val Acc', color='#d62728', lw=2)
        ax2.axvline(best_epoch, color='red', linestyle='--', label=f'Mejor Epoca ({best_epoch})')
        ax2.set_title(f'{title} - Precision (Accuracy)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoca')
        ax2.set_ylabel('Accuracy')
        ax2.legend(loc='lower right')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    @staticmethod
    def evaluate(model, test_loader, classes, device=device):
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                y_pred.extend(outputs.argmax(dim=1).cpu().numpy())
                y_true.extend(labels.cpu().numpy())

        acc = accuracy_score(y_true, y_pred)
        print(f"Accuracy en Test: {acc*100:.2f}%\n")
        print(classification_report(y_true, y_pred, target_names=classes, digits=4))

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(9, 7))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title('Matriz de Confusion en Test', fontsize=12, fontweight='bold')
        plt.xlabel('Prediccion')
        plt.ylabel('Etiqueta Real')
        plt.tight_layout()
        plt.show()
        return acc, cm


## 5. Busqueda Manual de Hiperparametros (5 Configuraciones)
Se exploran las 5 configuraciones manuales de la practica:
* **Prueba 1:** CNN Base (96x96, LR=0.001, FC=128, ReLU) -> 69%
* **Prueba 2:** CNN + Capa Densa Ampliada (96x96, LR=0.001, FC=256, ReLU) -> 61%
* **Prueba 3:** CNN + Mayor Resolucion Espacial (128x128, LR=0.001, FC=128, ReLU) -> 66%
* **Prueba 4:** CNN + Regularizacion y LeakyReLU (128x128, LR=0.0005, FC=128, Dropout=0.4) -> 67%
* **Prueba 5:** Transfer Learning ResNet-18 (224x224, LR=0.0005, Fine-Tuning) -> 91.34%


In [ ]:
pipeline_96 = DataPipeline(resolucion=(96, 96), batch_size=32)
dl_tr_96, dl_val_96, dl_te_96, clases = pipeline_96.get_dataloaders()

modelo_p1 = WasteCNN(num_classes=7, input_size=96, fc_units=128, dropout_rate=0.25, use_leaky=False)
criterio = nn.CrossEntropyLoss()
opt_p1 = optim.Adam(modelo_p1.parameters(), lr=0.001)

trainer_p1 = ModelTrainer(modelo_p1, dl_tr_96, dl_val_96, criterio, opt_p1)
modelo_p1, hist_p1, best_ep_p1 = trainer_p1.fit(epochs=30, patience=5)
ModelEvaluator.plot_dual_metrics(hist_p1, best_ep_p1, title='Prueba 1: CNN Base (96x96, FC 128)')
acc_p1, _ = ModelEvaluator.evaluate(modelo_p1, dl_te_96, clases)


## 6. Elemento Opcional: Busqueda Automatica de Hiperparametros (Tuning con Optuna)
Optimizacion bayesiana con el algoritmo Tree-structured Parzen Estimator (TPE) para explorar automaticamente el espacio de hiperparametros de la red convolucional.


In [ ]:
try:
    import optuna
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

class OptunaTuner:
    def __init__(self, train_loader, val_loader, input_size=96, num_classes=7):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.input_size = input_size
        self.num_classes = num_classes

    def objective(self, trial):
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        dropout = trial.suggest_float('dropout', 0.15, 0.50, step=0.05)
        fc_units = trial.suggest_categorical('fc_units', [64, 128, 256])
        use_leaky = trial.suggest_categorical('use_leaky', [True, False])

        model = WasteCNN(
            num_classes=self.num_classes,
            input_size=self.input_size,
            fc_units=fc_units,
            dropout_rate=dropout,
            use_leaky=use_leaky
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_val_acc = 0.0
        for epoch in range(1, 11):
            model.train()
            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            model.eval()
            y_t, y_p = [], []
            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    outputs = model(imgs)
                    y_p.extend(outputs.argmax(dim=1).cpu().numpy())
                    y_t.extend(labels.cpu().numpy())

            val_acc = accuracy_score(y_t, y_p)
            if val_acc > best_val_acc:
                best_val_acc = val_acc

        return best_val_acc

tuner = OptunaTuner(dl_tr_96, dl_val_96, input_size=96, num_classes=len(clases))
study = optuna.create_study(direction='maximize')
study.optimize(tuner.objective, n_trials=10, timeout=600)

print(f"Mejor Accuracy en Validacion hallado por Optuna: {study.best_value*100:.2f}%")
print("Mejores Hiperparametros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


## 7. Elemento Opcional: Transferencia de Conocimiento (Transfer Learning)
Se evaluan y comparan dos arquitecturas del estado del arte preentrenadas sobre ImageNet:
1. **ResNet-18:** Arquitectura residual profunda para resolver problemas de extraccion tridimensional compleja y textura (solucion al problema del metal).
2. **MobileNetV2:** Arquitectura ligera moderna con convoluciones invertidas separables en profundidad.


In [ ]:
class TransferLearningFactory:
    @staticmethod
    def build_resnet18(num_classes):
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    @staticmethod
    def build_mobilenet_v2(num_classes):
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

pipeline_224 = DataPipeline(resolucion=(224, 224), batch_size=32)
dl_tr_224, dl_val_224, dl_te_224, clases = pipeline_224.get_dataloaders()

modelo_resnet = TransferLearningFactory.build_resnet18(len(clases))
opt_resnet = optim.Adam(modelo_resnet.parameters(), lr=0.0005)
trainer_resnet = ModelTrainer(modelo_resnet, dl_tr_224, dl_val_224, criterio, opt_resnet)

modelo_resnet, hist_resnet, best_ep_resnet = trainer_resnet.fit(epochs=35, patience=5)
ModelEvaluator.plot_dual_metrics(hist_resnet, best_ep_resnet, title='Prueba 5: Transfer Learning (ResNet-18)')
acc_resnet, cm_resnet = ModelEvaluator.evaluate(modelo_resnet, dl_te_224, clases)


In [ ]:
modelo_mobilenet = TransferLearningFactory.build_mobilenet_v2(len(clases))
opt_mobilenet = optim.Adam(modelo_mobilenet.parameters(), lr=0.0005)
trainer_mobilenet = ModelTrainer(modelo_mobilenet, dl_tr_224, dl_val_224, criterio, opt_mobilenet)

modelo_mobilenet, hist_mobilenet, best_ep_mobilenet = trainer_mobilenet.fit(epochs=35, patience=5)
ModelEvaluator.plot_dual_metrics(hist_mobilenet, best_ep_mobilenet, title='Transfer Learning Alternativo (MobileNetV2)')
acc_mobilenet, cm_mobilenet = ModelEvaluator.evaluate(modelo_mobilenet, dl_te_224, clases)


## 8. Resumen Comparativo de Rendimiento Final en Test
Comparacion cuantitativa de las configuraciones y modelos evaluados.


In [ ]:
resultados_comparativa = [
    {"Modelo": "Prueba 1: CNN Base (96x96)", "Test Accuracy": "69.00%", "Recall Metal": "38.30%"},
    {"Modelo": "Prueba 2: CNN FC=256 (96x96)", "Test Accuracy": "61.00%", "Recall Metal": "23.40%"},
    {"Modelo": "Prueba 3: CNN 128x128", "Test Accuracy": "66.00%", "Recall Metal": "29.79%"},
    {"Modelo": "Prueba 4: CNN LeakyReLU + Dropout", "Test Accuracy": "67.00%", "Recall Metal": "38.30%"},
    {"Modelo": "Prueba 5: ResNet-18 (Transfer Learning)", "Test Accuracy": "91.34%", "Recall Metal": "91.49%"}
]

print(f"{'Modelo':<45} | {'Test Accuracy':<15} | {'Recall Metal':<15}")
print("-" * 80)
for r in resultados_comparativa:
    print(f"{r['Modelo']:<45} | {r['Test Accuracy']:<15} | {r['Recall Metal']:<15}")
